![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 04: LangChain Programming)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-AI-lab](https://github.com/tulip-lab/agentic-AI-lab/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 4C: Custom Tools, Storage and Controlled Actions

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local mock action agent with in-memory storage</td></tr>
<tr><td align="left">Optional part</td><td>LangChain tool wrappers if packages are available</td></tr>
<tr><td align="left">Main output</td><td>A controlled agent that can create, list and update simple support tickets safely</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m04c-overview)
2. [Setup and Background](#m04c-setup)
3. [Core Concepts](#m04c-storage-actions)
4. [Guided Implementation](#m04c-local-agent)
5. [Testing and Analysis](#m04c-testing)
6. [Student Tasks](#m04c-student-tasks)
7. [Submission and Reflection](#m04c-submission)

---

<a id="m04c-overview"></a>

### 1. Overview and Learning Goals

M04B introduced tool agents, but its tools were harmless: a calculator returns a number and forgets everything. M04C takes the next step by adding **controlled storage** and **controlled actions** — tools that change something that persists after the call.

This is an important step toward practical agentic AI. Many useful agents need to remember or update something: a support ticket, a task list, a draft, a note, a search result or a workflow state. But once an agent can update stored information, the safety requirements become stricter. A calculator mistake evaporates; a storage mistake is like writing in a shared logbook in pen — the wrong entry stays there for everyone who reads it later.

In this notebook, you will build a local teaching agent that manages simple support tickets in memory. It can create tickets, list tickets and update ticket status. It cannot read private files, send email, call the shell, access real databases, or modify external systems.

```text
User request
     |
     v
Intent router ------------------------------------------+
  |               |                |                     |
  | create ticket | list tickets   | update status       | unsafe request
  v               v                v                     v
Validate title  Read the        Validate ticket id     Refusal
+ description   local store     + status               (nothing changes)
  |               |                |
  +---------------+----------------+
                  |
                  v
        In-memory TicketStore
                  |
                  v
        Structured response
        (action label + answer)
```

Every path through this diagram ends in a structured response that names the action taken, and the only component allowed to touch the store is a validated, approved action. That discipline is the lesson of the session.

By the end of this session, you should be able to distinguish harmless calculation tools from state-changing tools, design a small controlled storage layer, validate actions before changing state, test normal and unsafe requests, and explain why this prepares for M05C LangGraph stateful workflows.

<a id="m04c-setup"></a>

### 2. Setup and Background

A chatbot answer can be wrong, but it usually disappears after the conversation. A storage action persists. If an agent creates the wrong ticket, deletes the wrong record, or updates a status incorrectly, the effect remains — and other people or programs may act on it before anyone notices. That is why state-changing tools need validation, logging and clear boundaries, in a way that a calculator never did.

In this notebook, storage is deliberately simple and local. We use an in-memory Python object rather than a real database: it disappears when the runtime restarts, so you can experiment freely, yet the *design pattern* you practise is the same one that protects real data. The table below shows where this sits on the risk ladder.

<div align="center">

<table>
<thead>
<tr><th><strong>Storage type</strong></th><th><strong>Example</strong></th><th><strong>Risk level</strong></th><th><strong>Use in this notebook</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">No storage</td><td>Calculator result</td><td>Low</td><td>Used in M04B.</td></tr>
<tr><td align="left">In-memory storage</td><td>Python list/dictionary while notebook runs</td><td>Low to moderate</td><td>Used in M04C.</td></tr>
<tr><td align="left">Local file storage</td><td>JSON or CSV file</td><td>Moderate</td><td>Optional extension only.</td></tr>
<tr><td align="left">External database</td><td>SQL, vector DB, CRM</td><td>Higher</td><td>Not used in this lab.</td></tr>
<tr><td align="left">External action system</td><td>Email, calendar, payment, shell</td><td>High</td><td>Not allowed in this lab.</td></tr>
</tbody>
</table>

</div>

This lab does not use real external side effects. The agent may only operate on the local in-memory ticket store created inside this notebook. Do not add file readers, shell commands, email senders, database writers, credential tools or private-data access — each of those belongs to a higher rung of the ladder and needs controls we have not built yet.

In [ ]:
# Standard-library imports only: the mandatory action agent must run with no
# installation, no API key and no internet access.
import json
import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

print("M04C setup complete.")

<a id="m04c-storage-actions"></a>

### 3. Core Concepts

#### 3.1 Storage and Controlled Actions

The teaching example is a small support-ticket system. It is not a real helpdesk; it is a safe local simulation with just enough structure to be realistic.

A ticket has:

```text
ticket_id: generated identifier (T001, T002, ...)
title: short issue title
description: issue detail
status: open / in_progress / resolved
```

The allowed actions are deliberately few and deliberately named:

```text
create_ticket(title, description)
list_tickets()
update_ticket_status(ticket_id, status)
```

The forbidden actions are just as explicit:

```text
delete all data
read private files
send email
run shell commands
access credentials
connect to real external systems
```

Notice what is missing from the allowed list: there is no delete action at all, and status can only move between three known values. Designing an action set is as much about what you leave out as what you include — an agent cannot misuse an operation that does not exist.

In [ ]:
ALLOWED_STATUSES = {"open", "in_progress", "resolved"}

@dataclass
class Ticket:
    ticket_id: str
    title: str
    description: str
    status: str = "open"


@dataclass
class TicketStore:
    """A simple in-memory ticket store for teaching controlled actions.

    Design decision: the store exposes exactly three named operations and no
    generic "write anything" method. State can only change through these
    controlled functions, never through free-form model text.
    """

    tickets: Dict[str, Ticket] = field(default_factory=dict)
    next_id: int = 1

    def create_ticket(self, title: str, description: str) -> Dict[str, Any]:
        # IDs are generated by the store, never chosen by the caller: this
        # prevents collisions and stops requests from spoofing identifiers.
        ticket_id = f"T{self.next_id:03d}"
        self.next_id += 1

        ticket = Ticket(
            ticket_id=ticket_id,
            title=title.strip(),
            description=description.strip(),
            status="open",
        )

        self.tickets[ticket_id] = ticket
        return {"ok": True, "error": None, "result": ticket.__dict__}

    def list_tickets(self) -> Dict[str, Any]:
        # Read-only action: no state change, so no validation is needed here.
        return {
            "ok": True,
            "error": None,
            "result": [ticket.__dict__ for ticket in self.tickets.values()],
        }

    def update_ticket_status(self, ticket_id: str, status: str) -> Dict[str, Any]:
        # The store re-checks its own invariants even though the agent also
        # validates: defence in depth means no single missed check corrupts state.
        if ticket_id not in self.tickets:
            return {"ok": False, "error": f"Unknown ticket_id: {ticket_id}", "result": None}

        if status not in ALLOWED_STATUSES:
            return {
                "ok": False,
                "error": f"Invalid status: {status}. Allowed: {sorted(ALLOWED_STATUSES)}",
                "result": None,
            }

        self.tickets[ticket_id].status = status
        return {"ok": True, "error": None, "result": self.tickets[ticket_id].__dict__}


store = TicketStore()
store.create_ticket("Cannot access Flowise", "The dashboard does not open on localhost.")
store.list_tickets()

The `TicketStore` is the storage layer. It is intentionally small, but it demonstrates a central idea: state-changing operations should be handled by controlled functions with clear contracts, not by free-form model text. Notice two design details in the output above. First, the ticket ID `T001` was generated by the store itself — callers never choose IDs, so they cannot collide or be spoofed. Second, the store re-validates status values even though the agent will validate them too; this "defence in depth" means one missed check elsewhere cannot corrupt the data.

In [ ]:
def validate_create_ticket_args(args: Dict[str, Any]) -> Dict[str, Any]:
    """Validate and clean the fields for create_ticket.

    On success the validator returns *cleaned* values (stripped strings), so
    the store always receives tidy data regardless of how the user typed it.
    """
    if not isinstance(args, dict):
        return {"ok": False, "error": "Arguments must be a dictionary.", "result": None}

    title = args.get("title")
    description = args.get("description")

    if not isinstance(title, str) or not title.strip():
        return {"ok": False, "error": "title must be a non-empty string.", "result": None}

    if not isinstance(description, str) or not description.strip():
        return {"ok": False, "error": "description must be a non-empty string.", "result": None}

    # Length cap: stored fields should have bounds, otherwise a single request
    # can bloat the store or break downstream displays.
    if len(title.strip()) > 120:
        return {"ok": False, "error": "title must be at most 120 characters.", "result": None}

    return {
        "ok": True,
        "error": None,
        "result": {
            "title": title.strip(),
            "description": description.strip(),
        },
    }


def validate_update_status_args(args: Dict[str, Any]) -> Dict[str, Any]:
    """Validate ticket_id format and status value for update_ticket_status."""
    if not isinstance(args, dict):
        return {"ok": False, "error": "Arguments must be a dictionary.", "result": None}

    ticket_id = args.get("ticket_id")
    status = args.get("status")

    if not isinstance(ticket_id, str) or not ticket_id.strip():
        return {"ok": False, "error": "ticket_id must be a non-empty string.", "result": None}

    # Normalise before checking: users type "t001" as often as "T001".
    ticket_id = ticket_id.strip().upper()

    # Format check catches typos early, before the store looks anything up.
    if not re.fullmatch(r"T\d{3}", ticket_id):
        return {"ok": False, "error": "ticket_id must use the format T001.", "result": None}

    # Closed set of statuses: an agent should never invent workflow states.
    if not isinstance(status, str) or status.strip() not in ALLOWED_STATUSES:
        return {
            "ok": False,
            "error": f"status must be one of {sorted(ALLOWED_STATUSES)}.",
            "result": None,
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "ticket_id": ticket_id,
            "status": status.strip(),
        },
    }


print(validate_create_ticket_args({"title": "RAG issue", "description": "Retriever returns weak context."}))
print(validate_update_status_args({"ticket_id": "T001", "status": "resolved"}))

Validation happens before storage is changed — the same pattern as M04B, but with higher stakes, because here an unvalidated action would leave a permanent wrong record rather than a wrong number. Note also what the validators *return*: on success, cleaned and normalised values (stripped whitespace, upper-cased IDs); on failure, an error that names the exact rule broken. Run the cell and check both printed lines show `ok: True` with tidy values. Then try `validate_update_status_args({"ticket_id": "T1", "status": "resolved"})` — the format check should reject it before the store is ever consulted.

<a id="m04c-local-agent"></a>

### 4. Guided Implementation

#### 4.1 Mandatory Local Action Agent

The local action agent routes text requests to approved actions. It is a transparent mock version of an agentic workflow: everything a real agent executor would decide invisibly is written out as readable Python.

The layering deserves attention, because it is the pattern you will reuse in every later lab:

```text
request text --> refusal check --> intent routing --> extract args
                                                          |
                                                          v
                                             validate args (reject bad input)
                                                          |
                                                          v
                                             store action (the only state change)
```

A real LLM could replace the keyword-based intent routing and argument extraction, and in production systems it often does. But the refusal check, the validation and the controlled store stay in ordinary code either way — the program, not the model, decides which actions are allowed.

In [ ]:
def extract_create_ticket_args(text: str) -> Dict[str, Any]:
    """Extract simple title/description fields from text.

    Expected pattern:
    create ticket title: ... description: ...

    This stands in for the language understanding a real LLM would provide.
    A fixed pattern keeps the lab deterministic and testable; swapping in a
    model later would change only this function, not the safety layers.
    """

    if not isinstance(text, str):
        return {}

    # The title capture stops at "description:" (or end of text), so both
    # fields can live in one request line.
    title_match = re.search(r"title\s*:\s*(.+?)(?:\s+description\s*:|$)", text, flags=re.IGNORECASE)
    desc_match = re.search(r"description\s*:\s*(.+)$", text, flags=re.IGNORECASE)

    args = {}
    if title_match:
        args["title"] = title_match.group(1).strip()
    if desc_match:
        args["description"] = desc_match.group(1).strip()

    return args


def extract_update_status_args(text: str) -> Dict[str, Any]:
    """Extract ticket_id and status from text.

    Expected pattern:
    update T001 status resolved
    """

    if not isinstance(text, str):
        return {}

    # Only well-formed IDs (T + three digits) and known statuses are captured;
    # anything else is left for the validator to report as missing.
    ticket_match = re.search(r"\b(T\d{3})\b", text, flags=re.IGNORECASE)
    status_match = re.search(r"\b(open|in_progress|resolved)\b", text, flags=re.IGNORECASE)

    args = {}
    if ticket_match:
        args["ticket_id"] = ticket_match.group(1).upper()
    if status_match:
        args["status"] = status_match.group(1).lower()

    return args


print(extract_create_ticket_args("create ticket title: RAG issue description: Retriever returns weak context."))
print(extract_update_status_args("update T001 status resolved"))

In [ ]:
class LocalActionAgent:
    """A controlled local action agent for ticket management.

    Every response carries an "action" label (create_ticket, list_tickets,
    update_ticket_status, validation_error, refuse or direct_response), so a
    reader or a test can always tell what KIND of thing happened, not just
    read the answer text.
    """

    def __init__(self, store: TicketStore):
        self.store = store

    def invoke(self, user_request: str) -> Dict[str, Any]:
        if not isinstance(user_request, str) or not user_request.strip():
            return {"ok": False, "error": "user_request must be a non-empty string.", "result": None}

        lower = user_request.lower()

        # Safety check FIRST, before any intent routing: a request mixing an
        # unsafe action with a legitimate one ("read private file and create
        # ticket") must be refused outright, not partially served.
        unsafe_keywords = [
            "read private file", "private file", "shell", "terminal", "command",
            "send email", "password", "api key", "credential", "delete all",
            "drop database", "external database"
        ]

        if any(keyword in lower for keyword in unsafe_keywords):
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "refuse",
                    "answer": "I cannot perform private-file access, shell commands, email sending, credential access, deletion, or external-system actions in this lab.",
                },
            }

        # Intent: create. Extract -> validate -> only then touch the store.
        if "create ticket" in lower:
            args = extract_create_ticket_args(user_request)
            validation = validate_create_ticket_args(args)
            if not validation["ok"]:
                return {
                    "ok": True,
                    "error": None,
                    "result": {
                        "action": "validation_error",
                        "answer": validation["error"],
                    },
                }

            created = self.store.create_ticket(**validation["result"])
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "create_ticket",
                    "ticket": created["result"],
                    "answer": f"Created ticket {created['result']['ticket_id']} with status open.",
                },
            }

        # Intent: list. Read-only, so no validation layer is needed.
        if "list tickets" in lower or "show tickets" in lower:
            tickets = self.store.list_tickets()["result"]
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "list_tickets",
                    "tickets": tickets,
                    "answer": f"There are {len(tickets)} ticket(s) in the local store.",
                },
            }

        # Intent: update. Two failure layers - argument validation, then the
        # store's own check for unknown IDs - and both report the same way.
        if "update" in lower and "status" in lower:
            args = extract_update_status_args(user_request)
            validation = validate_update_status_args(args)
            if not validation["ok"]:
                return {
                    "ok": True,
                    "error": None,
                    "result": {
                        "action": "validation_error",
                        "answer": validation["error"],
                    },
                }

            update = self.store.update_ticket_status(**validation["result"])
            if not update["ok"]:
                return {
                    "ok": True,
                    "error": None,
                    "result": {
                        "action": "validation_error",
                        "answer": update["error"],
                    },
                }

            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "update_ticket_status",
                    "ticket": update["result"],
                    "answer": f"Updated ticket {update['result']['ticket_id']} to {update['result']['status']}.",
                },
            }

        # Default: explain capabilities honestly instead of guessing an action.
        return {
            "ok": True,
            "error": None,
            "result": {
                "action": "direct_response",
                "answer": (
                    "This local agent can create tickets, list tickets, and update ticket status. "
                    "Use patterns such as: create ticket title: ... description: ...; list tickets; update T001 status resolved."
                ),
            },
        }


ticket_store = TicketStore()
action_agent = LocalActionAgent(ticket_store)

result = action_agent.invoke("create ticket title: Flowise login issue description: Student cannot open the Flowise dashboard.")
result

In [ ]:
def display_action_result(agent_result: Dict[str, Any]) -> None:
    # Read the action label first, then the answer: "what did the agent DO?"
    # comes before "what did it SAY?".
    if not agent_result.get("ok"):
        print("ERROR:", agent_result.get("error"))
        return

    result = agent_result["result"]
    print("Action:", result.get("action"))
    print("Answer:", result.get("answer"))

    if "ticket" in result:
        print("Ticket:", json.dumps(result["ticket"], indent=2))
    if "tickets" in result:
        print("Tickets:", json.dumps(result["tickets"], indent=2))


display_action_result(result)
display_action_result(action_agent.invoke("list tickets"))

The action agent returns structured information about the action taken, which makes it possible to see at a glance whether it created, listed, updated, refused or returned a validation error. In a real deployment this record would go to a log, because for state-changing agents the audit trail — which action ran, with which arguments, against which record — matters as much as the answer shown to the user.

<a id="m04c-optional-langchain"></a>

#### 4.2 Optional LangChain Tool Wrappers

This section is optional. It shows how the local functions you just built could be registered as LangChain tools if the package is available. It does not call an LLM and does not require an API key unless you later connect these tools to a real model.

The registration flow looks like this:

```text
Plain Python function            LangChain tool                  Agent / model
+----------------------+  @tool  +----------------------+  bind  +------------------+
| create_ticket(...)   | ------> | name, description,   | -----> | model can see    |
| validate, then store |         | args schema, func    |        | and propose the  |
+----------------------+         +----------------------+        | tool call        |
                                                                  +------------------+
```

The `@tool` decorator reads the function's name, signature and docstring and produces the same kind of record you built by hand as `ToolSpec` in M04B. Run this section only if package installation is allowed in your environment; otherwise record it as skipped.

In [ ]:
# Optional installation cell.
# Commented out on purpose so "Run all" never installs packages as a side
# effect. Uncomment only if package installation is allowed.

# !pip install -q langchain langchain-core

In [ ]:
# Optional LangChain tool wrappers.
# This cell is safe: it skips with an explanation if langchain_core is missing.

def optional_create_langchain_tools() -> Dict[str, Any]:
    try:
        from langchain_core.tools import tool
    except ImportError as exc:
        return {"ok": False, "error": f"langchain_core is not installed: {exc}", "result": None}

    # A fresh store, so the optional demo cannot disturb the mandatory one.
    local_store = TicketStore()

    # Note that the wrapper calls the SAME validator before the SAME store
    # method: wrapping a function as a tool must never bypass its checks.
    @tool
    def create_ticket_tool(title: str, description: str) -> str:
        """Create a local support ticket with a title and description."""
        validation = validate_create_ticket_args({"title": title, "description": description})
        if not validation["ok"]:
            return validation["error"]
        created = local_store.create_ticket(**validation["result"])
        return json.dumps(created["result"])

    @tool
    def list_tickets_tool() -> str:
        """List local support tickets."""
        return json.dumps(local_store.list_tickets()["result"])

    return {
        "ok": True,
        "error": None,
        "result": [create_ticket_tool, list_tickets_tool],
    }


optional_tools = optional_create_langchain_tools()
optional_tools

The optional wrapper section shows the relationship between local Python functions and LangChain tools: the decorator changes how a function is *described and discovered*, not what it is allowed to do. The central safety pattern is unchanged — validate before action, restrict tool scope, and avoid private or external side effects unless they are explicitly authorised and tested. If the cell reported that `langchain_core` is not installed, that is fine; note it as skipped and move on.

<a id="m04c-testing"></a>

### 5. Testing and Analysis

A controlled action agent needs tests for every outcome type: successful actions (create, list, update), invalid inputs (missing fields, bad status values), unknown records, safe direct responses and unsafe requests. The cell below uses a **fresh store** so results do not depend on what you ran earlier in the notebook — for stateful systems, test isolation is itself a lesson: a test that depends on leftover state passes or fails for the wrong reasons.

Run the cell. Success prints one line; a failure raises `AssertionError` at the first broken behaviour, and the comment above that line names the guarantee that was lost.

In [ ]:
# A fresh store isolates these tests from anything created earlier in the
# notebook. Each block protects one behaviour; the first failing assert stops
# the cell, so fix failures top to bottom.
test_store = TicketStore()
test_agent = LocalActionAgent(test_store)

# Normal: create ticket (first ticket in a fresh store must be T001).
create = test_agent.invoke("create ticket title: RAG issue description: Retriever returns weak context.")
assert create["ok"] is True
assert create["result"]["action"] == "create_ticket"
assert create["result"]["ticket"]["ticket_id"] == "T001"
assert create["result"]["ticket"]["status"] == "open"

# Normal: list tickets reflects the state change just made.
list_result = test_agent.invoke("list tickets")
assert list_result["ok"] is True
assert list_result["result"]["action"] == "list_tickets"
assert len(list_result["result"]["tickets"]) == 1

# Normal: update status through the full extract-validate-store path.
update = test_agent.invoke("update T001 status resolved")
assert update["ok"] is True
assert update["result"]["action"] == "update_ticket_status"
assert update["result"]["ticket"]["status"] == "resolved"

# Failure: missing description is caught before the store changes.
missing_desc = test_agent.invoke("create ticket title: Missing description")
assert missing_desc["ok"] is True
assert missing_desc["result"]["action"] == "validation_error"

# Failure: invalid status ("closed" is not in the allowed set).
bad_status = test_agent.invoke("update T001 status closed")
assert bad_status["ok"] is True
assert bad_status["result"]["action"] == "validation_error"

# Failure: unknown ticket id passes format checks but fails in the store.
unknown_ticket = test_agent.invoke("update T999 status resolved")
assert unknown_ticket["ok"] is True
assert unknown_ticket["result"]["action"] == "validation_error"

# Boundary: unsafe request mixed with a legitimate one is refused outright.
private_file = test_agent.invoke("read private file and create ticket")
assert private_file["ok"] is True
assert private_file["result"]["action"] == "refuse"

# Boundary: shell command is refused.
shell = test_agent.invoke("run shell command to inspect local files")
assert shell["ok"] is True
assert shell["result"]["action"] == "refuse"

# Direct response: unsupported but safe request gets help, not a guessed action.
direct = test_agent.invoke("What can you do?")
assert direct["ok"] is True
assert direct["result"]["action"] == "direct_response"

# Invalid input: empty request is an input error, not an agent action.
invalid = test_agent.invoke("")
assert invalid["ok"] is False

print("All M04C mandatory local-action tests passed.")

In [ ]:
# Inspect representative results with a fresh store: watch the action label
# change across create, list, update, refusal and deletion-refusal cases.

demo_store = TicketStore()
demo_agent = LocalActionAgent(demo_store)

requests = [
    "create ticket title: API key safety description: A student asks where to store API keys.",
    "list tickets",
    "update T001 status in_progress",
    "update T001 status resolved",
    "send email to the teacher",
    "delete all tickets",
]

for request in requests:
    print("\nUSER:", request)
    display_action_result(demo_agent.invoke(request))

Notice the difference between an accepted action, a validation error, a direct response and a refusal — four different outcomes, each with its own label, and only the first one changed any state. The last two requests are especially instructive: "send email" and "delete all" both sound routine, yet both were refused because the action set simply does not contain them. In a later LangGraph workflow, these outcomes become separate branches in the graph, which is why labelling them cleanly now pays off.

<a id="m04c-student-tasks"></a>

### 6. Student Tasks

Complete the tasks below. The mandatory local action agent must run without external API calls. Tasks 2 to 5 are programming tasks, so your work must demonstrate normal, edge and failure behaviour as described in the table.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run all cells through the mandatory testing section in a fresh runtime and confirm all tests pass.</td><td>Establishes a known-good baseline for a stateful system before you change its schema.</td><td>Output showing <code>All M04C mandatory local-action tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add priority field</td><td>Extend tickets with a <code>priority</code> field whose allowed values are <code>low</code>, <code>medium</code> and <code>high</code>, defaulting to <code>medium</code>. Normal: a ticket created with an explicit priority stores it. Edge: a ticket created without a priority gets <code>medium</code>. Failure: nothing outside the allowed set is ever stored.</td><td>Extending a stored schema while keeping old requests working is the everyday reality of maintaining stateful systems.</td><td>Updated dataclass/store code.</td></tr>
<tr><td align="left">Task 3: Validate priority</td><td>Update the creation validator so that a supplied priority must be one of the allowed values, and a missing priority becomes <code>medium</code>. Failure: an invalid value such as <code>urgent</code> returns a validation error naming the allowed set.</td><td>A closed value set is the same discipline as <code>ALLOWED_STATUSES</code>: agents must not invent workflow states.</td><td>Printed validation examples for one valid, one defaulted and one rejected case.</td></tr>
<tr><td align="left">Task 4: Extend request parsing</td><td>Support a pattern such as <code>priority: high</code> in create-ticket requests. Edge: requests without the pattern must still parse correctly (and default to <code>medium</code> downstream).</td><td>Every new field needs an extraction rule, a validation rule and a storage rule — this task makes you touch all three layers consistently.</td><td>Updated extraction code plus one printed example.</td></tr>
<tr><td align="left">Task 5: Add tests</td><td>Add at least four <code>assert</code>-based tests: valid priority, default priority, invalid priority, and an unsafe-request refusal that still works after your changes.</td><td>Schema changes are where existing guarantees quietly break; tests prove the old boundary survived the new feature.</td><td>Test cell output showing all added tests pass.</td></tr>
<tr><td align="left">Task 6: Inspect outputs</td><td>Print two representative results: one successful ticket with a priority and one validation error or refusal.</td><td>Reading the action label and the stored record is how you audit a state-changing agent.</td><td>Readable printed outputs for both cases.</td></tr>
<tr><td align="left">Task 7: Optional LangChain wrapper</td><td>If packages are available, run the optional wrapper section. If not, write <code>Skipped: langchain_core not installed</code>.</td><td>Seeing your own functions become LangChain tools connects this lab to the wider ecosystem.</td><td>Tool wrapper output or an explicit skipped note.</td></tr>
<tr><td align="left">Task 8: Reflection</td><td>Write 150-250 words explaining why storage actions require stronger validation than calculator tools.</td><td>The risk-scaling argument, not the ticket system itself, is the transferable idea of this session.</td><td>150-250 word reflection in a markdown cell.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter (Tasks 2-5).
# Extend the ticket system with a priority field. Work layer by layer, in the
# same order the request travels: storage schema -> validation -> extraction
# -> tests. Changing them in this order means each layer can be tested as soon
# as it is written.
#
# Suggested design:
# ALLOWED_PRIORITIES = {"low", "medium", "high"}
#
# 1. Storage schema - add priority to Ticket (and pass it through the store):
#    priority: str = "medium"
#
# 2. Validation - update validate_create_ticket_args:
#    - if priority is missing, set it to "medium" (edge case: default applies)
#    - if priority is provided, require low / medium / high
#      (failure case: reject anything else, naming the allowed set)
#
# 3. Extraction - update extract_create_ticket_args:
#    - support "priority: high" (tip: mirror how title/description are captured,
#      and make sure title capture stops before "priority:" as well)
#
# 4. Tests - add at least four:
#    - create ticket with priority high (normal)
#    - create ticket without priority defaults to medium (edge)
#    - invalid priority is rejected with a named error (failure)
#    - unsafe request is still refused after your changes (boundary)

<a id="m04c-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your priority-field extension.
3. Updated validation logic.
4. Updated request parsing logic.
5. At least four added tests with assert statements.
6. Printed output for one successful priority ticket and one validation/refusal case.
7. Optional LangChain wrapper output or skipped note.
8. 150-250 word reflection.
```

#### Quality checks

Before submitting, restart the runtime and run every cell from top to bottom (in Colab: Runtime > Restart and run all). Then confirm that:

- every cell runs without unhandled exceptions;
- the mandatory test cell prints `All M04C mandatory local-action tests passed.`;
- tickets created without a priority default to `medium`, and invalid priorities are rejected;
- unsafe requests are still refused after your changes;
- your added tests use `assert` statements, run against a fresh store, and all pass;
- no real API key or other secret appears anywhere in the notebook.

#### Debugging guide

- `T001` assertions fail unexpectedly: your test is reusing a store that already contains tickets, so IDs continue from where earlier cells left off. Create a fresh `TicketStore()` at the top of your test cell.
- Priority is always `medium` even when you supply one: your extraction function is not capturing the `priority:` pattern, so the validator only ever sees the default. Print the extractor's output on your exact request string.
- The title swallows the priority text (e.g. title becomes `"RAG issue priority: high"`): the title regex stops at `description:` but not at `priority:`; extend its stop pattern.
- `status must be one of [...]` when updating: check spelling — the allowed set is exactly `open`, `in_progress` (with underscore), `resolved`.
- `Unknown ticket_id` for a ticket you created: the ticket lives in a different store instance. Make sure the agent and your test share the same `TicketStore`.
- `AssertionError` in a test cell: the failing line names the behaviour; re-run that single request with `display_action_result(...)` and read the `Action:` label to see which path was taken.

#### Reflection questions

1. Why is a storage action riskier than a calculator tool?
2. Why should validation happen before storage is changed?
3. What is the difference between a validation error and a refusal?
4. Why are external databases, email and shell commands excluded from this lab?
5. How does this notebook prepare for M05C LangGraph stateful workflows?

#### Further Readings

- LangChain tools documentation: <https://python.langchain.com/docs/concepts/tools/>
- LangChain tool calling: <https://python.langchain.com/docs/concepts/tool_calling/>
- LangChain agents overview: <https://python.langchain.com/docs/concepts/agents/>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>